In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
from docling.document_converter import DocumentConverter


def norm_list(xs):
    return [str(x).replace("\u00a0", " ").strip() for x in xs]


def header_similarity(a, b):
    a = norm_list(a)
    b = norm_list(b)
    n = max(1, min(len(a), len(b)))
    return sum(a[i] == b[i] for i in range(n)) / n


def looks_like_data(cols):
    s = " ".join(norm_list(cols))
    return bool(re.search(r"\d", s)) or ("%" in s)


# minden oldal elejen leveszi a headert ahol nem kell oda
def fix_df(df, global_header):
    df = df.copy()
    cols = norm_list(df.columns.tolist())
    gh = norm_list(global_header)

    if header_similarity(cols, gh) >= 0.9:
        df.columns = gh
        return df

    # ha az oszlopnevek inkabb adatnak neznek ki, mentsuk vissza oket elso sornak.
    if looks_like_data(cols):
        row_df = pd.DataFrame([cols], columns=gh[: len(cols)])
        df.columns = gh[: len(df.columns)]
        out = pd.concat([row_df, df], ignore_index=True)
        out.columns = gh[: len(out.columns)]
        return out

    df.columns = gh[: len(df.columns)]
    return df


# a cegnevet es a cimet szetszedi
def split_name_address(text):
    if pd.isna(text):
        return pd.NA, pd.NA

    s = str(text).strip()
    # 4 számjegy + szóköz + 1 szó + vessző
    m = re.search(r"\b(\d{4}\s+\S+,\s*.*)", s)
    if m:
        address = m.group(1).strip()
        name = s[: m.start()].strip()
        return name, address

    return s, pd.NA


# arbevetelt adatait formazza
def hu_to_float(x):
    if x is None or pd.isna(x):
        return pd.NA

    s = str(x).strip()
    if s in ["N/A", "NA", "-", ""]:
        return pd.NA

    s = s.replace(" ", "").replace("\u00a0", "")
    s = s.replace(".", "")
    s = s.replace(",", ".")
    return s


# kiveszi a nevet a tablazat elott
def extract_name_from_doc(doc):
    md = doc.document.export_to_markdown()
    for line in md.splitlines():
        line = line.strip()
        if line.startswith("##"):
            return re.sub(r"^#+", "", line).strip()
    return pd.NA

In [ ]:
def pdf_to_dataframe(pdf_path: Path, converter: DocumentConverter) -> pd.DataFrame:
    doc = converter.convert(str(pdf_path))

    dfs = [tbl.export_to_dataframe(doc.document) for tbl in doc.document.tables]
    if not dfs:
        return pd.DataFrame()

    global_header = norm_list(dfs[0].columns.tolist())
    fixed = [fix_df(d, global_header) for d in dfs]

    big = pd.concat(fixed, ignore_index=True)

    # üres sorok dobasa
    big = big.dropna(how="all")
    big = big[(big.astype(str).apply(lambda r: "".join(r).strip(), axis=1) != "")]

    # nev kinyerese és beszurasa
    person_name = extract_name_from_doc(doc)
    big.insert(0, "Név", person_name)

    # Érdekeltség -> Cégnév, Cím
    if "Érdekeltség" in big.columns:
        big[["Cégnév", "Cím"]] = big["Érdekeltség"].apply(
            lambda x: pd.Series(split_name_address(x))
        )
        big = big.drop(columns=["Érdekeltség"])

    if "Árbevétel (2024) (eFt)" in big.columns:
        big["Árbevétel (2024) (eFt)"] = pd.to_numeric(
            big["Árbevétel (2024) (eFt)"].map(hu_to_float), errors="coerce"
        )

    if "Tulajdonosi lánc hossza 2" in big.columns:
        big["Tulajdonosi lánc hossza 2"] = pd.to_numeric(
            big["Tulajdonosi lánc hossza 2"], errors="coerce"
        )

    if "Befolyás mértéke 1" in big.columns:
        big["Befolyás mértéke 1"] = pd.to_numeric(
            big["Befolyás mértéke 1"]
            .astype("string")
            .str.replace("%", "", regex=False)
            .str.strip()
            .replace("", pd.NA),
            errors="coerce",
        )

    return big


def batch_pdf_to_excel(input_dir: str | Path, output_dir: str | Path) -> list[Path]:
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # környezeti beállítások
    os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
    os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"
    os.environ["HF_HOME"] = str(Path.home() / "hf_cache")
    os.environ["TRANSFORMERS_CACHE"] = str(Path.home() / "hf_cache")

    converter = DocumentConverter()

    pdf_files = sorted(input_dir.glob("*.pdf"))
    written = []

    for pdf_path in pdf_files:
        try:
            df = pdf_to_dataframe(pdf_path, converter)

            out_path = output_dir / f"{pdf_path.stem}.xlsx"
            with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
                df.to_excel(writer, index=False, sheet_name="Adatok")

            written.append(out_path)
            print(f"[OK] {pdf_path.name} -> {out_path.name}  (sorok: {len(df)})")

        except Exception as e:
            print(f"[HIBA] {pdf_path.name}: {e}")

    return written


batch_pdf_to_excel("input_pdf", "output_excel")

In [ ]:
data = pd.concat(
    [
        pd.read_excel(file, engine="calamine")
        for file in sorted(Path("output_excel").glob("*.xlsx"))
    ]
)

In [ ]:
data.to_excel("merged_entities.xlsx", index=False)